In [28]:
import pandas as pd
import re

# 파일 경로
station_path = "../data/서울교통공사_역주소 및 전화번호_20250318.csv"
sigungu_path = "../data/sigungu_code_info.csv"

SAVE_PATH = "../data"

# 인코딩 처리
try:
    df_station = pd.read_csv(station_path, encoding="utf-8")
except:
    df_station = pd.read_csv(station_path, encoding="cp949")

df_sigungu = pd.read_csv(sigungu_path, encoding="utf-8")

print(df_station.shape)
df_station.head()

(289, 7)


,연번,역번호,호선,역명,역전화번호,도로명주소,지번주소
0,1,150,1,서울,02-6110-1331,서울특별시 중구 세종대로 지하2(남대문로 5가),서울특별시 중구 남대문로5가 73-6 서울역(1호선)
1,2,151,1,시청,02-6110-1321,서울특별시 중구 세종대로 지하101(정동),서울특별시 중구 정동 5-5 시청역(1호선)
2,3,152,1,종각,02-6110-1311,서울특별시 종로구 종로 지하55(종로1가),서울특별시 종로구 종로1가 54 종각역(1호선)
3,4,153,1,종로3가,02-6110-1301,서울특별시 종로구 종로 지하129(종로3가),서울특별시 종로구 종로3가 10-5 종로3가역(1호선)
4,5,154,1,종로5가,02-6110-1291,서울특별시 종로구 종로 지하216(종로5가),서울특별시 종로구 종로5가 82-1 종로5가역(1호선)


In [29]:
# 자치구 칼럼 생성
def extract_gu(address):
    if pd.isna(address):
        return None
    match = re.search(r"서울특별시\s+(\S+구)", str(address))
    return match.group(1) if match else None

df_station["gu"] = df_station["도로명주소"].apply(extract_gu)

df_station[["도로명주소", "gu"]].head()

,도로명주소,gu
0,서울특별시 중구 세종대로 지하2(남대문로 5가),중구
1,서울특별시 중구 세종대로 지하101(정동),중구
2,서울특별시 종로구 종로 지하55(종로1가),종로구
3,서울특별시 종로구 종로 지하129(종로3가),종로구
4,서울특별시 종로구 종로 지하216(종로5가),종로구


In [30]:
# 법정동코드 매핑
# 서울(11로 시작)만 남김
df_sigungu_seoul = df_sigungu[
    df_sigungu["SIGUNGU_CD"].astype(str).str.startswith("11")
].copy()

df_sigungu_seoul = df_sigungu_seoul.rename(columns={
    "SIGUNGU_NM": "gu",
    "SIGUNGU_CD": "law_code"
})

df = df_station.merge(df_sigungu_seoul[["gu", "law_code"]],
                      on="gu",
                      how="left")

df["law_code"] = pd.to_numeric(df["law_code"], errors="coerce").astype("Int64")

df["law_code"].dtype

Int64Dtype()

In [31]:
# name_std 생성
# 역 이름은 부제를 제외하고, '역'까지 포함하여 표기
def make_name_std(name):
    if pd.isna(name):
        return None
    base = re.sub(r"\(.*?\)", "", str(name)).strip()
    if not base.endswith("역"):
        base = base + "역"
    return base

df["name_std"] = df["역명"].apply(make_name_std)

df[["역명", "name_std"]].head()

,역명,name_std
0,서울,서울역
1,시청,시청역
2,종각,종각역
3,종로3가,종로3가역
4,종로5가,종로5가역


In [32]:
from dotenv import load_dotenv
import os

load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
print(GOOGLE_API_KEY[:6] + "..." if GOOGLE_API_KEY else "❌ 키 로드 실패")

AIzaSy...


In [33]:
df["lat"] = None
df["lon"] = None

import requests
import time

def geocode_address(address, api_key):
    if pd.isna(address):
        return None, None, "NO_ADDRESS"
    
    url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {
        "address": address,
        "key": api_key
    }
    
    try:
        response = requests.get(url, params=params, timeout=10)
        data = response.json()
        
        status = data.get("status")
        
        if status == "OK":
            location = data["results"][0]["geometry"]["location"]
            return location.get("lat"), location.get("lng"), "OK"
        
        else:
            return None, None, status
    
    except Exception:
        return None, None, "ERROR"

df["geo_status"] = None

In [34]:
from tqdm import tqdm

for idx in tqdm(df.index):

    # 이미 처리 완료된 행은 skip
    if pd.notna(df.at[idx, "geo_status"]):
        continue

    lat, lng, status = geocode_address(
        df.at[idx, "도로명주소"],
        GOOGLE_API_KEY
    )

    df.at[idx, "lat"] = lat
    df.at[idx, "lon"] = lng
    df.at[idx, "geo_status"] = status

    # quota 초과 시 중단
    if status == "OVER_QUERY_LIMIT":
        print("API quota 초과. 중단.")
        break

    time.sleep(0.2)


100%|██████████| 289/289 [03:53<00:00,  1.24it/s]


In [35]:
# latitude / longitude 제거
df = df.drop(columns=["latitude", "longitude"], errors="ignore")

# lat/lon만 유지
df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
df["lon"] = pd.to_numeric(df["lon"], errors="coerce")

In [36]:
output_file = os.path.join(SAVE_PATH, "서울교통공사_역정보.csv")

df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

output_file

'../data\\서울교통공사_역정보.csv'